# 🎬 Movie Recommendation System
**By: Vraj Patoliya | Gandhinagar University | B.Tech IT**

A content-based recommendation system using CountVectorizer + Cosine Similarity.

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import ast
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
from nltk.stem.porter import PorterStemmer
import nltk
nltk.download('punkt', quiet=True)

print('All libraries imported!')
print(f'Pandas: {pd.__version__} | NumPy: {np.__version__}')

## Step 2: Load Dataset
> Download from: https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata

In [ ]:
movies  = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

print('Movies shape :', movies.shape)
print('Credits shape:', credits.shape)

# Merge on title
movies = movies.merge(credits, on='title')
print('After merge  :', movies.shape)
movies.head(2)

## Step 3: Select Columns & Clean

In [ ]:
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]
print('Missing values:')
print(movies.isnull().sum())
movies.dropna(inplace=True)
print('Shape after cleaning:', movies.shape)

## Step 4: EDA — Genre Distribution

In [ ]:
def extract_names(obj):
    return [i['name'] for i in ast.literal_eval(obj)]

all_genres = [g for sub in movies['genres'].apply(extract_names) for g in sub]
genre_counts = Counter(all_genres).most_common(12)
genres_df = pd.DataFrame(genre_counts, columns=['Genre','Count'])

plt.figure(figsize=(12,5))
sns.barplot(data=genres_df, x='Genre', y='Count', palette='Blues_d')
plt.title('Top 12 Movie Genres in Dataset', fontsize=14, fontweight='bold')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('genre_distribution.png', dpi=150)
plt.show()
print('Chart saved!')

In [ ]:
movies['overview_length'] = movies['overview'].apply(lambda x: len(x.split()))
plt.figure(figsize=(10,4))
sns.histplot(movies['overview_length'], bins=30, color='steelblue', kde=True)
plt.title('Movie Overview Word Count Distribution', fontsize=13, fontweight='bold')
plt.xlabel('Word Count')
plt.tight_layout()
plt.savefig('overview_length.png', dpi=150)
plt.show()
print(f'Avg overview length: {movies["overview_length"].mean():.1f} words')

## Step 5: Feature Engineering

In [ ]:
def convert(obj):
    return [i['name'] for i in ast.literal_eval(obj)]

def get_top3_cast(obj):
    return [i['name'] for i in ast.literal_eval(obj)][:3]

def get_director(obj):
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            return [i['name']]
    return []

movies['genres']   = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast']     = movies['cast'].apply(get_top3_cast)
movies['crew']     = movies['crew'].apply(get_director)
movies['overview'] = movies['overview'].apply(lambda x: x.split())

print('Conversion done!')
print('Sample genres:', movies['genres'].iloc[0])
print('Sample cast  :', movies['cast'].iloc[0])
print('Director     :', movies['crew'].iloc[0])

In [ ]:
# Remove spaces from names to avoid false matches
def collapse(lst):
    return [i.replace(' ','') for i in lst]

movies['genres']   = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)
movies['cast']     = movies['cast'].apply(collapse)
movies['crew']     = movies['crew'].apply(collapse)

print('Spaces collapsed!')
print('Cast after collapse:', movies['cast'].iloc[0])

In [ ]:
# Combine all features into one 'tags' column
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

new_df = movies[['movie_id','title','tags']].copy()
new_df['tags'] = new_df['tags'].apply(lambda x: ' '.join(x).lower())

print('Tags column created!')
print('Sample tags:')
print(new_df.iloc[0]['tags'][:250])
new_df.head(3)

## Step 6: Apply Stemming

In [ ]:
ps = PorterStemmer()

def stem(text):
    return ' '.join([ps.stem(word) for word in text.split()])

new_df['tags'] = new_df['tags'].apply(stem)
print('Stemming applied!')
print('Sample stemmed tags:')
print(new_df.iloc[0]['tags'][:200])

## Step 7: Vectorization — CountVectorizer

In [ ]:
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()

print(f'Vector shape: {vectors.shape}')
print(f'Total features: {len(cv.get_feature_names_out())}')

# Top 20 frequent words
word_freq = vectors.sum(axis=0)
words = cv.get_feature_names_out()
top20 = sorted(zip(words, word_freq), key=lambda x: -x[1])[:20]
print('\nTop 20 words in tags:')
for w, f in top20:
    print(f'  {w:20s} {int(f)}')

## Step 8: Cosine Similarity Matrix

In [ ]:
similarity = cosine_similarity(vectors)
print(f'Similarity matrix: {similarity.shape}')
print(f'Total pairs: {similarity.shape[0]*similarity.shape[1]:,}')

# Heatmap for first 15 movies
plt.figure(figsize=(12,10))
subset = similarity[:15,:15]
labels = [t[:15]+'...' if len(t)>15 else t for t in new_df['title'].values[:15]]
sns.heatmap(subset, xticklabels=labels, yticklabels=labels,
            cmap='Blues', annot=True, fmt='.2f', annot_kws={'size':7})
plt.title('Cosine Similarity Heatmap (First 15 Movies)', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('similarity_heatmap.png', dpi=150)
plt.show()

## Step 9: Recommendation Function

In [ ]:
def recommend(movie, n=5):
    """
    Recommend top-n similar movies.
    Args:
        movie (str): Title of movie
        n     (int): Number of recommendations
    Returns:
        list of (title, score) tuples
    """
    matches = new_df[new_df['title'].str.lower() == movie.lower()]
    if matches.empty:
        print(f'Movie "{movie}" not found!')
        return []
    
    idx = matches.index[0]
    distances = similarity[idx]
    sorted_movies = sorted(enumerate(distances), key=lambda x: x[1], reverse=True)[1:n+1]
    
    print(f'\nMovies similar to "{movie}":')
    print('-' * 48)
    results = []
    for rank, (i, score) in enumerate(sorted_movies, 1):
        title = new_df.iloc[i].title
        print(f'  {rank}. {title:<38} score: {score:.3f}')
        results.append((title, round(score, 3)))
    return results

print('recommend() function ready!')

## Step 10: Test Recommendations

In [ ]:
recommend('Avatar')

In [ ]:
recommend('The Dark Knight')

In [ ]:
recommend('Inception')

In [ ]:
recommend('The Godfather')

In [ ]:
recommend('Interstellar', n=10)

## Step 11: Visualization

In [ ]:
def visualize_recommendations(movie, n=8):
    matches = new_df[new_df['title'].str.lower() == movie.lower()]
    if matches.empty:
        print(f'Movie "{movie}" not found!')
        return
    idx = matches.index[0]
    distances = similarity[idx]
    sorted_movies = sorted(enumerate(distances), key=lambda x: x[1], reverse=True)[1:n+1]
    titles = [new_df.iloc[i].title[:20] for i,_ in sorted_movies]
    scores = [round(s*100,1) for _,s in sorted_movies]
    plt.figure(figsize=(10,5))
    colors = plt.cm.Blues_r(np.linspace(0.3,0.9,len(titles)))
    bars = plt.barh(titles[::-1], scores[::-1], color=colors)
    for bar, score in zip(bars, scores[::-1]):
        plt.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
                 f'{score}%', va='center', fontsize=10, fontweight='bold')
    plt.title(f'Top {n} Recommendations for "{movie}"', fontsize=13, fontweight='bold')
    plt.xlabel('Similarity Score (%)')
    plt.xlim(0, max(scores)+10)
    plt.tight_layout()
    fname = f'reco_{movie.replace(" ","_")}.png'
    plt.savefig(fname, dpi=150)
    plt.show()
    print(f'Chart saved as {fname}')

visualize_recommendations('Avatar')
visualize_recommendations('The Dark Knight')

## Step 12: Save Model for Streamlit App

In [ ]:
pickle.dump(new_df,     open('movies.pkl',     'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))
pickle.dump(cv,         open('vectorizer.pkl', 'wb'))

print('Model saved!')
print('  movies.pkl     — movie dataframe')
print('  similarity.pkl — cosine similarity matrix')
print('  vectorizer.pkl — fitted CountVectorizer')
print()
print('Run Streamlit app with:  streamlit run app.py')

## Step 13: Summary

In [ ]:
print('='*50)
print('  MOVIE RECOMMENDATION SYSTEM SUMMARY')
print('='*50)
print(f'  Total movies        : {len(new_df)}')
print(f'  Feature vector size : {vectors.shape[1]}')
print(f'  Similarity matrix   : {similarity.shape[0]} x {similarity.shape[1]}')
print(f'  Algorithm           : Cosine Similarity')
print(f'  Vectorizer          : CountVectorizer (5000 features)')
print(f'  Features used       : Overview + Genres + Keywords + Cast + Director')
print('='*50)
print('  Author  : Vraj Patoliya')
print('  College : Gandhinagar University — B.Tech IT')
print('='*50)